In [1]:
!pip install datasets nltk spacy scikit-learn gensim matplotlib seaborn
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     -- ------------------------------------- 0.8/12.8 MB 1.2 MB/s eta 0:00:10
     ---- ----------------------------------- 1.3/12.8 MB 1.7 MB/s eta 0:00:07
     ----- ---------------------------------- 1.8/12.8 MB 2.0 MB/s eta 0:00:06
     -------- ------------------------------- 2.6/12.8 MB 2.2 MB/s eta 0:00:05
     --------- ------------------------------ 2.9/12.8 MB 2.2 MB/s eta 0:00:05
     ---------- ----------------------------- 3.4/12.8 MB 2.1 MB/s eta 0:00:05
     ------------ --------------------------- 3.9/12.8 MB 2.2 MB/s eta 0:00:05
     ------------- -------------------------- 4.5/12.8 MB 2.2 MB/s eta 0:00:04
     ---------------- ----------------------- 5.2/12.8 MB 2.3 MB/s eta 0:00:04
 

In [2]:
import re
import nltk
import spacy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
from collections import Counter
nltk.download('stopwords')

C:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [3]:
nlp = spacy.load("en_core_web_sm")
stop_words = set(stopwords.words('english'))

In [4]:
dataset = load_dataset("amazon_polarity", split="train[:5000]")
df = pd.DataFrame(dataset)
df = df[['content']]  # text column
df.rename(columns={'content': 'text'}, inplace=True)

C:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\datasets--amazon_polarity. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|████████████████████████████████████████| 400000/400000 [00:00<00:0

In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)

In [ ]:
def preprocess(text):
    doc = nlp(text)
    tokens = [
        token.lemma_ for token in doc
        if token.text not in stop_words and token.is_alpha
    ]
    return tokens

df['tokens'] = df['clean_text'].apply(preprocess)

In [ ]:
vocab = set()
for tokens in df['tokens']:
    vocab.update(tokens)

print(f"Vocabulary size: {len(vocab)}")

In [ ]:
df['processed_text'] = df['tokens'].apply(lambda x: " ".join(x))

bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(df['processed_text'])

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['processed_text'])

In [ ]:
w2v_model = Word2Vec(
    sentences=df['tokens'],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

print(w2v_model.wv.most_similar("good"))

In [ ]:
def sentence_embedding(tokens):
    vectors = [
        w2v_model.wv[word] for word in tokens if word in w2v_model.wv
    ]
    if len(vectors) == 0:
        return np.zeros(100)
    return np.mean(vectors, axis=0)

df['sentence_embedding'] = df['tokens'].apply(sentence_embedding)
sentence_matrix = np.vstack(df['sentence_embedding'].values)

In [ ]:
def find_similar(text, top_n=5):
    tokens = preprocess(clean_text(text))
    query_vec = sentence_embedding(tokens).reshape(1, -1)

    similarities = cosine_similarity(query_vec, sentence_matrix)[0]
    top_indices = similarities.argsort()[-top_n:][::-1]

    return df.iloc[top_indices][['text']]

print(find_similar("this product is amazing and works great"))

In [ ]:
all_words = [word for tokens in df['tokens'] for word in tokens]
common_words = Counter(all_words).most_common(20)

words, counts = zip(*common_words)

plt.figure()
plt.bar(words, counts)
plt.xticks(rotation=45)
plt.title("Top Words")
plt.show()

In [ ]:
sim_matrix = cosine_similarity(sentence_matrix[:100])
sns.heatmap(sim_matrix)
plt.title("Similarity Heatmap")
plt.show()